In [44]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

In [45]:
os.listdir()

['.ipynb_checkpoints',
 'BasketBall_Mania.ipynb',
 'march-machine-learning-mania-2026',
 'Submission.csv',
 'Untitled.ipynb']

In [46]:
os.listdir('march-machine-learning-mania-2026')

['Cities.csv',
 'Conferences.csv',
 'MConferenceTourneyGames.csv',
 'MGameCities.csv',
 'MMasseyOrdinals.csv',
 'MNCAATourneyCompactResults.csv',
 'MNCAATourneyDetailedResults.csv',
 'MNCAATourneySeedRoundSlots.csv',
 'MNCAATourneySeeds.csv',
 'MNCAATourneySlots.csv',
 'MRegularSeasonCompactResults.csv',
 'MRegularSeasonDetailedResults.csv',
 'MSeasons.csv',
 'MSecondaryTourneyCompactResults.csv',
 'MSecondaryTourneyTeams.csv',
 'MTeamCoaches.csv',
 'MTeamConferences.csv',
 'MTeams.csv',
 'MTeamSpellings.csv',
 'SampleSubmissionStage1.csv',
 'SampleSubmissionStage2.csv',
 'WConferenceTourneyGames.csv',
 'WGameCities.csv',
 'WNCAATourneyCompactResults.csv',
 'WNCAATourneyDetailedResults.csv',
 'WNCAATourneySeeds.csv',
 'WNCAATourneySlots.csv',
 'WRegularSeasonCompactResults.csv',
 'WRegularSeasonDetailedResults.csv',
 'WSeasons.csv',
 'WSecondaryTourneyCompactResults.csv',
 'WSecondaryTourneyTeams.csv',
 'WTeamConferences.csv',
 'WTeams.csv',
 'WTeamSpellings.csv']

In [47]:
# Load key Datasets

teams = pd.read_csv('march-machine-learning-mania-2026/MTeams.csv')
seeds = pd.read_csv('march-machine-learning-mania-2026/MNCAATourneySeeds.csv')
season_results = pd.read_csv('march-machine-learning-mania-2026/MRegularSeasonCompactResults.csv')
tourney_results = pd.read_csv('march-machine-learning-mania-2026/MNCAATourneyCompactResults.csv')
print(teams.shape)
print(seeds.shape)
print(season_results.shape)
print(tourney_results.shape)

(381, 4)
(2626, 3)
(196823, 8)
(2585, 8)


In [48]:
# Look at each Dataset

In [49]:
print("TEAMS:")
print(teams.head())
print("\nSEEDS:")
print(seeds.head())
print("\nSEASON RESULTS")
print(season_results.head())
print("\nTOURNEY RESULTS")
print(tourney_results.head())

TEAMS:
   TeamID     TeamName  FirstD1Season  LastD1Season
0    1101  Abilene Chr           2014          2026
1    1102    Air Force           1985          2026
2    1103        Akron           1985          2026
3    1104      Alabama           1985          2026
4    1105  Alabama A&M           2000          2026

SEEDS:
   Season Seed  TeamID
0    1985  W01    1207
1    1985  W02    1210
2    1985  W03    1228
3    1985  W04    1260
4    1985  W05    1374

SEASON RESULTS
   Season  DayNum  WTeamID  WScore  LTeamID  LScore WLoc  NumOT
0    1985      20     1228      81     1328      64    N      0
1    1985      25     1106      77     1354      70    H      0
2    1985      25     1112      63     1223      56    H      0
3    1985      25     1165      70     1432      54    H      0
4    1985      25     1192      86     1447      74    H      0

TOURNEY RESULTS
   Season  DayNum  WTeamID  WScore  LTeamID  LScore WLoc  NumOT
0    1985     136     1116      63     1234      54   

In [50]:
# Calculate win rate for each team per season

In [51]:
# Step 1: Count wins per team per season
wins = season_results.groupby(['Season','WTeamID']).size().reset_index(name="Wins")
wins.columns = ['Season', 'TeamID', 'Wins']

In [52]:
# Step 2: Count losses per team per season
losses = season_results.groupby(['Season','LTeamID']).size().reset_index(name="Losses")
losses.columns = ['Season', 'TeamID', 'Losses']

In [53]:
# Step 3: Merge wins and losses
team_stats = wins.merge(losses, on=['Season', 'TeamID'], how='outer').fillna(0)

In [54]:
# Step 4: Calculate the win rate
team_stats['WinRate'] = team_stats['Wins'] / (team_stats['Wins'] + team_stats['Losses'])
print(team_stats.head(10))

   Season  TeamID  Wins  Losses   WinRate
0    1985    1102   5.0    19.0  0.208333
1    1985    1103   9.0    14.0  0.391304
2    1985    1104  21.0     9.0  0.700000
3    1985    1106  10.0    14.0  0.416667
4    1985    1108  19.0     6.0  0.760000
5    1985    1109   1.0    23.0  0.041667
6    1985    1110   7.0    18.0  0.280000
7    1985    1111  10.0    14.0  0.416667
8    1985    1112  18.0     9.0  0.666667
9    1985    1113  11.0    16.0  0.407407


In [55]:
# Add seed information and build our prediction features

In [56]:
# Clean seed column
seeds['SeedNum'] = seeds['Seed'].str.extract('(\d+)').astype(int)

In [57]:
# Merge seed info with team stats
team_stats = team_stats.merge(seeds[['Season', 'TeamID', 'SeedNum']], on=['Season', 'TeamID'], how='left')

In [58]:
# Fill missing seeds with 16 (worst seed = didn't qualify)
team_stats['SeedNum'].fillna(16, inplace=True)
print(team_stats.head(10))
print(team_stats.shape)

   Season  TeamID  Wins  Losses   WinRate  SeedNum
0    1985    1102   5.0    19.0  0.208333     16.0
1    1985    1103   9.0    14.0  0.391304     16.0
2    1985    1104  21.0     9.0  0.700000      7.0
3    1985    1106  10.0    14.0  0.416667     16.0
4    1985    1108  19.0     6.0  0.760000     16.0
5    1985    1109   1.0    23.0  0.041667     16.0
6    1985    1110   7.0    18.0  0.280000     16.0
7    1985    1111  10.0    14.0  0.416667     16.0
8    1985    1112  18.0     9.0  0.666667     10.0
9    1985    1113  11.0    16.0  0.407407     16.0
(13753, 6)


In [59]:
avg_score = season_results.groupby(['Season', 'WTeamID'], as_index=False).agg(AvgPointsScored=('WScore', 'mean')).rename(columns={'WTeamID': 'TeamID'})
avg_allowed = season_results.groupby(['Season', 'LTeamID'], as_index=False).agg(AvgPointsScored=('WScore', 'mean')).rename(columns={'LTeamID': 'TeamID'})

In [60]:
# Merge into team stats
team_stats = team_stats.merge(avg_score, on=['Season', 'TeamID'], how='left')
team_stats = team_stats.merge(avg_allowed, on=['Season', 'TeamID'], how='left')
team_stats.fillna(team_stats.mean(numeric_only=True), inplace=True)
print(team_stats.columns.tolist())

['Season', 'TeamID', 'Wins', 'Losses', 'WinRate', 'SeedNum', 'AvgPointsScored_x', 'AvgPointsScored_y']


In [70]:
# Training Data for our model
# Building training dataset from tournament results
# For each game, creating a row with features of both teams

def build_features(row, team_stats):
    season = row['Season']
    w_team = row['WTeamID']
    l_team = row['LTeamID']
    
    w_stats = team_stats[(team_stats['Season'] == season) & (team_stats['TeamID'] == w_team)]
    l_stats = team_stats[(team_stats['Season'] == season) & (team_stats['TeamID'] == l_team)]
    
    if w_stats.empty or l_stats.empty:
        return None
    
    w_stats = w_stats.iloc[0]
    l_stats = l_stats.iloc[0]
    
    return {
        'SeedDiff': w_stats['SeedNum'] - l_stats['SeedNum'],
        'WinRateDiff': w_stats['WinRate'] - l_stats['WinRate'],
        'WinsDiff': w_stats['Wins'] - l_stats['Wins'],
        'AvgScoreDiff': w_stats['AvgPointsScored_x'] - l_stats['AvgPointsScored_x'],
        'AvgAllowedDiff': w_stats['AvgPointsScored_y'] - l_stats['AvgPointsScored_y'],
        'Result': 1
    }

df_train = pd.DataFrame(rows)

In [71]:
# Apply to all tournament games
rows = []
for _, game in tourney_results.iterrows():
    features = build_features(game, team_stats)
    if features:
        rows.append(features)

df_train = pd.DataFrame(rows)
print(df_train.shape)
print(df_train.head())

(2585, 6)
   SeedDiff  WinRateDiff  WinsDiff  AvgScoreDiff  AvgAllowedDiff  Result
0       1.0    -0.030303       1.0     -7.297619        4.883333       1
1       5.0    -0.059310       1.0      2.607843       -5.693182       1
2     -15.0     0.546616      14.0      1.952727       -5.611111       1
3       1.0     0.062169       1.0      1.252632        6.682540       1
4     -11.0     0.025926       3.0     11.202174        0.857143       1


In [72]:
# Currently all results are 1 (winner perspective)
# We need to add losing perspective too (result = 0)
# This doubles our data and makes model learn both sides

# Add losing perspective
df_loss = df_train.copy()
df_loss['SeedDiff'] = -df_train['SeedDiff']
df_loss['WinRateDiff'] = -df_train['WinRateDiff']
df_loss['WinsDiff'] = -df_train['WinsDiff']
df_loss['AvgScoreDiff'] = -df_train['AvgScoreDiff']
df_loss['AvgAllowedDiff'] = -df_train['AvgAllowedDiff']
df_loss['Result'] = 0

In [73]:
# Combining Both
df_model = pd.concat([df_train, df_loss], ignore_index=True)
print(df_model.shape)
print(df_model.head(-10))

(5170, 6)
      SeedDiff  WinRateDiff  WinsDiff  AvgScoreDiff  AvgAllowedDiff  Result
0          1.0    -0.030303       1.0     -7.297619        4.883333       1
1          5.0    -0.059310       1.0      2.607843       -5.693182       1
2        -15.0     0.546616      14.0      1.952727       -5.611111       1
3          1.0     0.062169       1.0      1.252632        6.682540       1
4        -11.0     0.025926       3.0     11.202174        0.857143       1
...        ...          ...       ...           ...             ...     ...
5155       4.0    -0.030303      -1.0     -7.793333      -10.805556       0
5156       3.0    -0.264706      -9.0      4.634897        1.500000       0
5157       3.0    -0.124777      -5.0     -1.860000       -9.375000       0
5158       7.0    -0.151515      -5.0      0.190000       -2.807692       0
5159       4.0    -0.113191      -3.0     -4.560000       -1.666667       0

[5160 rows x 6 columns]


In [74]:
# Training our model (we will try 2 different model, Log loss is the scoring metric use in this competition)

In [75]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, log_loss

In [76]:
# Train
X = df_model[['SeedDiff', 'WinRateDiff', 'WinsDiff', 'AvgScoreDiff', 'AvgAllowedDiff']]
y = df_model['Result']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [77]:
# Model 1: Logistic Regression
lr = LogisticRegression()
lr.fit(X_train, y_train)
lr_prob = lr.predict_proba(X_test)[:, 1]

print("Accuracy:", accuracy_score(y_test, lr.predict(X_test)))
print("Log Loss:", log_loss(y_test, lr_prob))

Accuracy: 0.7369439071566731
Log Loss: 0.5346460175108744


In [78]:
# Model 2: Gradient Boosting Classifier
gb = GradientBoostingClassifier(n_estimators=100, random_state=23)
gb.fit(X_train, y_train)
gb_prob = gb.predict_proba(X_test)[:,1]
print("Gradient Boosting Classifier:")
print(" Accuracy Score:", accuracy_score(y_test, gb.predict(X_test)))
print(" Log Loss:", log_loss(y_test, gb_prob))

Gradient Boosting Classifier:
 Accuracy Score: 0.7330754352030948
 Log Loss: 0.5397573190919637


In [79]:
sub = pd.read_csv('March-machine-learning-mania-2026/SampleSubmissionStage1.csv')
print(sub.head())
print(sub.shape)

               ID  Pred
0  2022_1101_1102   0.5
1  2022_1101_1103   0.5
2  2022_1101_1104   0.5
3  2022_1101_1105   0.5
4  2022_1101_1106   0.5
(519144, 2)


In [84]:
sub['Season'] = sub['ID'].apply(lambda x: int(x.split('_')[0]))
sub['Team1'] = sub['ID'].apply(lambda x: int(x.split('_')[1]))
sub['Team2'] = sub['ID'].apply(lambda x: int(x.split('_')[2]))

# Merge team1 stats
sub = sub.merge(team_stats, left_on=['Season', 'Team1'], right_on=['Season', 'TeamID'], how='left')
sub = sub.rename(columns={
    'SeedNum': 'T1_Seed', 'WinRate': 'T1_WinRate', 'Wins': 'T1_Wins',
    'AvgPointsScored_x': 'T1_AvgScore', 'AvgPointsScored_y': 'T1_AvgAllowed'
})

# Merge team2 stats
sub = sub.merge(team_stats, left_on=['Season', 'Team2'], right_on=['Season', 'TeamID'], how='left', suffixes=('', '_t2'))
sub = sub.rename(columns={
    'SeedNum': 'T2_Seed', 'WinRate': 'T2_WinRate', 'Wins': 'T2_Wins',
    'AvgPointsScored_x': 'T2_AvgScore', 'AvgPointsScored_y': 'T2_AvgAllowed'
})

# Fill missing values
sub.fillna(0, inplace=True)

# Create feature differences
X_sub = pd.DataFrame({
    'SeedDiff': sub['T1_Seed'] - sub['T2_Seed'],
    'WinRateDiff': sub['T1_WinRate'] - sub['T2_WinRate'],
    'WinsDiff': sub['T1_Wins'] - sub['T2_Wins'],
    'AvgScoreDiff': sub['T1_AvgScore'] - sub['T2_AvgScore'],
    'AvgAllowedDiff': sub['T1_AvgAllowed'] - sub['T2_AvgAllowed']
})

In [85]:
# Predict and save
sub['Pred'] = lr.predict_proba(X_sub)[:, 1]
final_sub = sub[['ID', 'Pred']]
final_sub.to_csv('submission.csv', index=False)
print("Done!")
print(final_sub.head())

Done!
               ID      Pred
0  2022_1101_1102  0.613894
1  2022_1101_1103  0.373535
2  2022_1101_1104  0.179279
3  2022_1101_1105  0.561561
4  2022_1101_1106  0.567731
